# Notebook 9 – Categorical Data Analysis
**File:** `09_Categorical_Analysis.ipynb`

## Learning goals
Learn to inspect categories, count their frequencies, spot rare categories, understand cardinality and imbalance, and compare categories with a target variable.

## Why categorical analysis matters
Categorical columns describe groups such as department, payment method, product type, or customer segment. Before analysis or machine learning, we should understand which groups exist, how common they are, whether labels are inconsistent, and whether groups relate to the outcome.

The data below is synthetic and intended for practice.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.DataFrame({
    "customer_id": range(1, 13),
    "region": ["North", "South", "East", "West", "North", "South",
               "North", "East", "North", "South", "West", "North"],
    "payment_method": ["Card", "Cash", "Card", "UPI", "Card", "UPI",
                       "Card", "Cash", "Card", "UPI", "Other", "Card"],
    "product": ["A", "B", "A", "C", "A", "B", "A", "D", "A", "B", "E", "A"],
    "churned": ["No", "Yes", "No", "Yes", "No", "No",
                "No", "Yes", "No", "Yes", "Yes", "No"]
})
df

## 1. Unique Categories

**Definition:** The distinct labels present in a categorical column.

**Simple example:** `["Red", "Blue", "Red"]` has two unique categories: Red and Blue.

**Real-world example:** A hospital may list unique appointment types such as check-up, follow-up, and emergency.

**Business example:** A shop checks the unique payment methods customers used.

**AI/ML use case:** Identify valid labels before encoding categorical features for a model.

**Why it matters:** Unexpected values such as `"north"`, `"North "` and `"NORTH"` may represent the same group but be treated as separate categories.

In [ ]:
print("Unique regions:", df["region"].unique())
print("Number of unique regions:", df["region"].nunique())

# Example of inconsistent labels
messy_regions = pd.Series(["North", "north", "North ", "South"])
print("Before cleaning:", messy_regions.unique())
clean_regions = messy_regions.str.strip().str.title()
print("After cleaning:", clean_regions.unique())

## 2. Category Frequencies

**Definition:** The number of rows belonging to each category.

**Simple example:** In `["Cat", "Dog", "Cat"]`, Cat appears twice and Dog once.

**Real-world example:** Count the number of patients by appointment type.

**Business example:** Count orders by payment method to understand customer preferences.

**AI/ML use case:** Frequency counts can reveal rare labels and possible class imbalance.

**Why it matters:** Counts help identify the most common and least common groups before making comparisons.

In [ ]:
region_counts = df["region"].value_counts()
print(region_counts)
print("\nProportions:")
print(df["region"].value_counts(normalize=True).round(3))

region_counts.plot(kind="bar", edgecolor="black", title="Number of Customers by Region")
plt.xlabel("Region")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

## 3. Category Distribution

**Definition:** The proportion or percentage of observations in each category.

**Simple example:** If 6 of 10 orders use a card, cards represent 60% of orders.

**Real-world example:** Percentage of hospital visits by department.

**Business example:** Percentage of revenue or orders contributed by each product category. (For revenue share, sum revenue rather than count rows.)

**AI/ML use case:** Compare category proportions between training and test data to spot distribution shifts.

**Why it matters:** Percentages make group sizes easier to compare, especially when datasets have different totals.

In [ ]:
category_distribution = df["payment_method"].value_counts(normalize=True).mul(100).round(1)
print(category_distribution)

category_distribution.plot(kind="bar", edgecolor="black", title="Payment Method Distribution (%)")
plt.xlabel("Payment method")
plt.ylabel("Percent")
plt.tight_layout()
plt.show()

## 4. Rare Categories

**Definition:** Categories that appear only a small number of times or have a very small share of the data. There is no universal cutoff; choose one based on the dataset and task.

**Simple example:** In 100 orders, a payment type used once is rare.

**Real-world example:** A travel site may have a destination selected by only a few customers.

**Business example:** A retailer may have products with very few sales.

**AI/ML use case:** Rare categories can create unreliable estimates or sparse one-hot encoded columns. They may be grouped into `"Other"` when it makes business sense.

**Why it matters:** Rare groups may be noise, emerging opportunities, or important exceptions. Investigate before combining or removing them.

In [ ]:
counts = df["product"].value_counts()
rare_categories = counts[counts < 3]
print("All product counts:")
print(counts)
print("\nCategories appearing fewer than 3 times:")
print(rare_categories)

# Demonstration only: group categories with fewer than 3 rows into Other
df["product_grouped"] = df["product"].where(df["product"].map(counts) >= 3, "Other")
print("\nGrouped product counts:")
print(df["product_grouped"].value_counts())

## 5. High Cardinality

**Definition:** A categorical feature with many distinct categories compared with the number of rows.

**Simple example:** A column of unique order IDs has nearly one category per row.

**Real-world example:** A website may store thousands of distinct search terms.

**Business example:** A customer table may contain many unique postal codes or product IDs.

**AI/ML use case:** One-hot encoding a high-cardinality column can create thousands of columns. Alternatives include frequency encoding, carefully designed target encoding, hashing, embeddings, or domain-based grouping.

**Why it matters:** High cardinality can increase memory use and overfitting risk. IDs often identify records rather than provide generalizable meaning.

In [ ]:
high_cardinality = pd.DataFrame({
    "customer_id": [f"C{i:03}" for i in range(1, 11)],
    "region": ["North", "South"] * 5
})
for column in high_cardinality.columns:
    print(column, "unique categories =", high_cardinality[column].nunique(),
          "| rows =", len(high_cardinality))

## 6. Low Cardinality

**Definition:** A categorical feature with only a few distinct categories.

**Simple example:** A column with `"Yes"` and `"No"` has low cardinality.

**Real-world example:** A form may record whether a user accepted terms.

**Business example:** A customer may be assigned to one of a few membership tiers.

**AI/ML use case:** Low-cardinality columns are often straightforward to one-hot encode or represent as binary features, depending on meaning.

**Why it matters:** These columns are easy to summarize and visualize. Still, check that the labels are valid and consistent.

In [ ]:
for column in ["churned", "payment_method", "region"]:
    print(f"{column}: {df[column].nunique()} unique categories")

## 7. Category Imbalance

**Definition:** A situation where some categories occur much more often than others. In classification, target imbalance means one target class is much more common than another.

**Simple example:** 95 `"No"` values and 5 `"Yes"` values are imbalanced.

**Real-world example:** Fraudulent transactions are usually much less common than legitimate transactions.

**Business example:** Only a small fraction of customers may cancel a subscription.

**AI/ML use case:** An imbalanced target can make accuracy misleading. Consider precision, recall, F1-score, PR-AUC, class weights, or resampling within the training data, depending on the problem.

**Why it matters:** A model that always predicts the majority class may achieve high accuracy while failing to identify the minority class. Use metrics aligned with the real business cost.

In [ ]:
print("Target counts:")
print(df["churned"].value_counts())
print("\nTarget percentages:")
print(df["churned"].value_counts(normalize=True).mul(100).round(1))

df["churned"].value_counts().plot(kind="bar", edgecolor="black", title="Target Class Counts")
plt.xlabel("Churned")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

## 8. Relationship between categorical features and a target

**Definition:** Compare the target outcome across groups of a categorical feature.

**Simple example:** Compare the proportion of `"Yes"` outcomes for each region.

**Real-world example:** Compare readmission rates across hospital departments, while accounting for other relevant factors.

**Business example:** Compare customer churn rates by payment method or region.

**AI/ML use case:** Such analysis can reveal predictive signals, data leakage, or group differences. An association does not prove causation.

**Why it matters:** Raw counts alone can mislead when group sizes differ. Use within-group percentages as well as counts.

In [ ]:
# Cross-tabulation: counts of churn outcome for each region
counts_by_region = pd.crosstab(df["region"], df["churned"])
print("Counts by region:")
print(counts_by_region)

# Row percentages: within each region, what proportion churned?
rates_by_region = pd.crosstab(df["region"], df["churned"], normalize="index").mul(100).round(1)
print("\nPercentages within each region:")
print(rates_by_region)

rates_by_region.plot(kind="bar", stacked=True, edgecolor="black",
                     title="Churn Outcome by Region (%)")
plt.xlabel("Region")
plt.ylabel("Percent")
plt.legend(title="Churned")
plt.tight_layout()
plt.show()

## 9. Practical checklist for categorical data

1. Inspect unique labels and clean whitespace/case inconsistencies.
2. Check counts and percentages.
3. Investigate rare categories before grouping them.
4. Check cardinality before encoding.
5. Inspect target-class balance.
6. Compare categorical features with the target using counts and within-group rates.
7. Avoid target leakage: only use information available at prediction time.
8. Remember that association does not prove cause.

## Mini practice
1. What does `nunique()` return?
2. How is frequency different from proportion?
3. Why can high cardinality cause problems for one-hot encoding?
4. Why can accuracy be misleading on an imbalanced target?
5. What does `pd.crosstab(..., normalize="index")` show?

In [ ]:
# Optional answers — try the questions first.
answers = {
    1: "The number of distinct values.",
    2: "Frequency is a count; proportion is the count divided by the total.",
    3: "It can create a very large number of sparse columns and increase overfitting risk.",
    4: "A model can predict the majority class and still miss the minority class.",
    5: "Row-wise proportions, so each row sums to 1 (or 100 after multiplying by 100)."
}
answers